In [1]:
from datetime import datetime, timezone
from pathlib import Path
from functools import lru_cache

import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'fig1-heatmap-crosswalk-tables'

LOCAL_CROSSWALKS = {
    'azimuth': 'https://cdn.humanatlas.io/digital-objects/ctann/azimuth/v1.4/assets/azimuth-crosswalk.csv',
    'celltypist': 'https://cdn.humanatlas.io/digital-objects/ctann/celltypist/v1.3/assets/celltypist-crosswalk.csv',
    'deepcelltypes': 'https://cdn.humanatlas.io/digital-objects/ctann/deepcelltypes/v1.2/assets/deepcelltypes-crosswalk.csv',
    'deepcelltypes_hubmap': 'https://cdn.humanatlas.io/digital-objects/ctann/deepcelltypes-hubmap/v1.2/assets/deepcelltypes-hubmap-crosswalk.csv',
    'frmatch': 'https://cdn.humanatlas.io/digital-objects/ctann/frmatch/v1.0/assets/frmatch-crosswalk.csv',
    'pan_human_azimuth': 'https://cdn.humanatlas.io/digital-objects/ctann/pan-human-azimuth/v1.2/assets/pan-human-azimuth-crosswalk.csv',
    'popv': 'https://cdn.humanatlas.io/digital-objects/ctann/popv/v1.4/assets/popv-crosswalk.csv',
    'ribca': 'https://cdn.humanatlas.io/digital-objects/ctann/ribca/v1.0/assets/ribca-crosswalk.csv',
    'stellar': 'https://cdn.humanatlas.io/digital-objects/ctann/stellar/v1.0/assets/stellar-crosswalk.csv',
    'vccf': 'https://cdn.humanatlas.io/digital-objects/ctann/vccf/v1.2/assets/vccf-crosswalk.csv',
}

SOURCE_LABELS = {
    'azimuth': 'Azimuth', 'celltypist': 'CellTypist',
    'deepcelltypes': 'DeepCell Types',
    'deepcelltypes_hubmap': 'DeepCell Types-HuBMAP',
    'frmatch': 'FR-Match', 'pan_human_azimuth': 'Pan-human Azimuth',
    'popv': 'popV', 'ribca': 'RIBCA', 'stellar': 'STELLAR',
    'vccf': 'CDE Spatial Omics',
}

@lru_cache(maxsize=None)
def get_organ_label(organ_id):
    if not organ_id or pd.isna(organ_id):
        return 'Unspecified'
    iri = f'http://purl.obolibrary.org/obo/{organ_id.replace(":", "_")}'
    try:
        response = requests.get(
            'https://www.ebi.ac.uk/ols4/api/ontologies/uberon/terms',
            params={'iri': iri}, timeout=30,
        )
        terms = response.json().get('_embedded', {}).get('terms', [])
        return terms[0].get('label', 'Unspecified') if terms else 'Unspecified'
    except requests.RequestException:
        return 'Unspecified'


In [2]:
# Read the ten crosswalk tables and apply the Figure 1 organ filter.
frames = []
for source, url in LOCAL_CROSSWALKS.items():
    frame = pd.read_csv(url, skiprows=10)
    frame['source'] = source
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)
df['Organ'] = df['Organ_ID'].map(get_organ_label)
unspecified_rows = (df['Organ'] == 'Unspecified').sum()
df = df.loc[df['Organ'] != 'Unspecified'].copy()
df['Tool'] = df['source'].map(SOURCE_LABELS)

print(f'Rows after filtering: {len(df):,}')
print(f'Rows excluded because organ is Unspecified: {unspecified_rows:,}')


Rows after filtering: 3,457
Rows excluded because organ is Unspecified: 9


In [ ]:
# Count each CL ID once per tool and organ.
tool_organ_cl_id_counts = (
    df.drop_duplicates(['Tool', 'Organ', 'CL_ID'])
      .groupby(['Tool', 'Organ'], as_index=False)['CL_ID']
      .nunique()
      .rename(columns={'CL_ID': 'Number of Unique CL ID'})
      .sort_values(['Tool', 'Organ'], kind='stable')
      .reset_index(drop=True)
)

display(tool_organ_cl_id_counts)

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir = OUTPUT_ROOT / timestamp
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'tool-organ-unique-cl-id-counts.csv'
tool_organ_cl_id_counts.to_csv(output_file, index=False)


,Tool,Organ,Number of Unique CL ID
0,Azimuth,adipose tissue,26
1,Azimuth,adrenal gland,12
2,Azimuth,blood,41
3,Azimuth,bone marrow,43
4,Azimuth,brain,8
...,...,...,...
72,popV,thymus,23
73,popV,tongue,11
74,popV,trachea,18
75,popV,urinary bladder,15


Saved C:\Users\yokong\.vscode\ctann-data-descriptor-supporting-information\outputs\fig1-heatmap-crosswalk-tables\20260824T183252Z\tool-organ-unique-cl-id-counts.csv
